# NOVA RETAIL — Shadow Inventory & Ghost SKUs

## Análisis Forense de Inventario Invisible
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Identificación de vulnerabilidades estructurales en el catálogo legacy  
**Objetivo:** Detectar productos presentes en SAP pero invisibles para AS400, cuantificar su exposición económica y evaluar su relación con los puntos ciegos de control del inventario.

---

### Propósito de este notebook
Este notebook analiza una de las vulnerabilidades más críticas del entorno NOVA RETAIL:

- productos que existen en SAP pero no en AS400
- mercancía de alto valor que puede circular sin trazabilidad completa
- riesgo de pérdida material facilitado por opacidad estructural de catálogo

### Preguntas clave
1. ¿Cuántos SKUs son invisibles para AS400?
2. ¿Qué valor económico concentran?
3. ¿Qué categorías o marcas dominan esa invisibilidad?
4. ¿Qué tan defendible es tratar este fenómeno como una vulnerabilidad material?

### Nota del analista
La invisibilidad de catálogo no prueba por sí sola desvío intencional.  
Pero sí crea una condición operativa donde la mercancía puede moverse con menor capacidad de control, conciliación y auditoría.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")
dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")

print("Datasets cargados correctamente.")
print("products_sap:", products_sap.shape)
print("products_as400:", products_as400.shape)
print("dispatches:", dispatches.shape)
print("receptions:", receptions.shape)

Datasets cargados correctamente.
products_sap: (14000, 6)
products_as400: (14000, 6)
dispatches: (265964, 11)
receptions: (265964, 11)


In [2]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True].copy()

print(f"Total Ghost SKUs: {len(apple_ghosts)}")
apple_ghosts.head()

Total Ghost SKUs: 22


,sku_as400,description_as400,category_as400,price_as400,sku_sap_reference,is_ghost
2102,NaN,NaN,NaN,NaN,SAP-0002103,True
2112,NaN,NaN,NaN,NaN,SAP-0002113,True
2114,NaN,NaN,NaN,NaN,SAP-0002115,True
2119,NaN,NaN,NaN,NaN,SAP-0002120,True
2126,NaN,NaN,NaN,NaN,SAP-0002127,True


In [3]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
].copy()

sap_ghost_details.head(20)

,sku_sap,description_sap,category_sap,brand,price_sap,supplier_id
2102,SAP-0002103,AIRPODS APPLE 512GB,TELEFONIA,APPLE,17422.81,PROV-001
2112,SAP-0002113,FUNDA APPLE 128GB,TELEFONIA,APPLE,9266.41,PROV-001
2114,SAP-0002115,FUNDA APPLE 128GB,TELEFONIA,APPLE,28059.41,PROV-004
2119,SAP-0002120,"FUNDA APPLE 55""",TELEFONIA,APPLE,26302.09,PROV-030
2126,SAP-0002127,IPHONE APPLE 512GB,TELEFONIA,APPLE,9220.79,PROV-047
2130,SAP-0002131,"CARGADOR APPLE 43""",TELEFONIA,APPLE,23478.72,PROV-034
2140,SAP-0002141,FUNDA APPLE 512GB,TELEFONIA,APPLE,11737.40,PROV-037
2145,SAP-0002146,"AIRPODS APPLE 55""",TELEFONIA,APPLE,27240.79,PROV-018
2150,SAP-0002151,FUNDA APPLE 128GB,TELEFONIA,APPLE,24974.43,PROV-001
2152,SAP-0002153,"FUNDA APPLE 55""",TELEFONIA,APPLE,5722.80,PROV-046


In [4]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
].copy()

sap_ghost_details.head(20)

,sku_sap,description_sap,category_sap,brand,price_sap,supplier_id
2102,SAP-0002103,AIRPODS APPLE 512GB,TELEFONIA,APPLE,17422.81,PROV-001
2112,SAP-0002113,FUNDA APPLE 128GB,TELEFONIA,APPLE,9266.41,PROV-001
2114,SAP-0002115,FUNDA APPLE 128GB,TELEFONIA,APPLE,28059.41,PROV-004
2119,SAP-0002120,"FUNDA APPLE 55""",TELEFONIA,APPLE,26302.09,PROV-030
2126,SAP-0002127,IPHONE APPLE 512GB,TELEFONIA,APPLE,9220.79,PROV-047
2130,SAP-0002131,"CARGADOR APPLE 43""",TELEFONIA,APPLE,23478.72,PROV-034
2140,SAP-0002141,FUNDA APPLE 512GB,TELEFONIA,APPLE,11737.40,PROV-037
2145,SAP-0002146,"AIRPODS APPLE 55""",TELEFONIA,APPLE,27240.79,PROV-018
2150,SAP-0002151,FUNDA APPLE 128GB,TELEFONIA,APPLE,24974.43,PROV-001
2152,SAP-0002153,"FUNDA APPLE 55""",TELEFONIA,APPLE,5722.80,PROV-046


In [5]:
sap_ghost_details[[
    "sku_sap",
    "description_sap",
    "category_sap",
    "brand",
    "price_sap"
]].sort_values("price_sap", ascending=False).head(20)

,sku_sap,description_sap,category_sap,brand,price_sap
2114,SAP-0002115,FUNDA APPLE 128GB,TELEFONIA,APPLE,28059.41
2223,SAP-0002224,"IPHONE APPLE 65""",TELEFONIA,APPLE,28036.94
2145,SAP-0002146,"AIRPODS APPLE 55""",TELEFONIA,APPLE,27240.79
2199,SAP-0002200,"AIRPODS APPLE 75""",TELEFONIA,APPLE,26969.63
2119,SAP-0002120,"FUNDA APPLE 55""",TELEFONIA,APPLE,26302.09
2225,SAP-0002226,"GALAXY BUDS APPLE 65""",TELEFONIA,APPLE,25457.07
2150,SAP-0002151,FUNDA APPLE 128GB,TELEFONIA,APPLE,24974.43
2130,SAP-0002131,"CARGADOR APPLE 43""",TELEFONIA,APPLE,23478.72
2224,SAP-0002225,"GALAXY BUDS APPLE 55""",TELEFONIA,APPLE,23059.68
2233,SAP-0002234,CARGADOR APPLE 256GB,TELEFONIA,APPLE,20342.50


In [6]:
ghost_summary = pd.DataFrame({
    "indicador": [
        "Total Ghost SKUs",
        "Valor promedio por SKU",
        "Valor máximo por SKU",
        "Valor mínimo por SKU",
        "Valor total estimado del catálogo invisible"
    ],
    "valor": [
        len(sap_ghost_details),
        sap_ghost_details["price_sap"].mean(),
        sap_ghost_details["price_sap"].max(),
        sap_ghost_details["price_sap"].min(),
        sap_ghost_details["price_sap"].sum()
    ]
})

ghost_summary

,indicador,valor
0,Total Ghost SKUs,22.000000
1,Valor promedio por SKU,18562.857273
2,Valor máximo por SKU,28059.410000
3,Valor mínimo por SKU,5722.800000
4,Valor total estimado del catálogo invisible,408382.860000


In [7]:
sap_ghost_details["category_sap"].value_counts()

category_sap
TELEFONIA    22
Name: count, dtype: int64

In [9]:
sap_ghost_details["brand"].value_counts()

brand
APPLE    22
Name: count, dtype: int64

In [10]:
ghost_dispatches = dispatches[
    dispatches["sku"].isin(ghost_sap_refs)
].copy()

print("Despachos de Ghost SKUs:", len(ghost_dispatches))
ghost_dispatches.head(20)

Despachos de Ghost SKUs: 427


,dispatch_id,store_id,sku,quantity_dispatched,dispatch_date,dispatch_time,route,truck_id,driver_id,authorized_by,cedis_system_record
1004,DSP-00001005,1,SAP-0002115,11,2024-03-05,05:52,SUR,TRUCK-011,DRV-005,MGARCI07,SAP
1182,DSP-00001183,1,SAP-0002200,7,2024-03-15,04:49,SUR,TRUCK-010,DRV-030,RCISNE04,SAP
1489,DSP-00001490,2,SAP-0002234,8,2024-01-02,05:47,SUR,TRUCK-010,DRV-029,RCISNE04,SAP
1894,DSP-00001895,2,SAP-0002170,7,2024-02-02,04:49,SUR,TRUCK-020,DRV-032,RCISNE04,SAP
1983,DSP-00001984,2,SAP-0002103,12,2024-02-09,05:30,SUR,TRUCK-021,DRV-014,JLOPEZ11,SAP
3333,DSP-00003334,3,SAP-0002127,4,2024-02-02,03:01,GOLFO,TRUCK-008,DRV-029,RCISNE04,SAP
3450,DSP-00003451,3,SAP-0002193,5,2024-02-09,04:53,GOLFO,TRUCK-025,DRV-036,MGARCI07,SAP
3522,DSP-00003523,3,SAP-0002120,12,2024-02-16,03:17,GOLFO,TRUCK-023,DRV-034,MGARCI07,SAP
3671,DSP-00003672,3,SAP-0002197,7,2024-02-23,03:14,GOLFO,TRUCK-025,DRV-028,RCISNE04,SAP
3734,DSP-00003735,3,SAP-0002193,6,2024-03-01,03:39,GOLFO,TRUCK-011,DRV-032,RCISNE04,SAP


In [11]:
ghost_dispatches["route"].value_counts()

route
PACIFICO    110
GOLFO       106
CENTRO       96
SUR          80
NORTE        35
Name: count, dtype: int64

In [12]:
ghost_dispatches["store_id"].nunique()

170

In [13]:
ghost_dispatch_summary = ghost_dispatches.groupby("store_id").agg(
    total_despachos=("dispatch_id", "count"),
    total_unidades=("quantity_dispatched", "sum")
).reset_index().sort_values("total_unidades", ascending=False)

ghost_dispatch_summary.head(20)

,store_id,total_despachos,total_unidades
61,68,9,62
103,116,9,59
31,32,7,57
127,142,5,49
139,155,4,42
2,3,6,37
87,99,3,36
7,8,6,35
33,35,5,35
83,94,5,34


In [14]:
ghost_dispatch_value = ghost_dispatches.merge(
    products_sap[["sku_sap", "price_sap", "description_sap", "brand", "category_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

ghost_dispatch_value["dispatch_value_mxn"] = (
    ghost_dispatch_value["quantity_dispatched"] * ghost_dispatch_value["price_sap"]
)

ghost_dispatch_value[[
    "dispatch_id",
    "store_id",
    "sku",
    "description_sap",
    "quantity_dispatched",
    "price_sap",
    "dispatch_value_mxn",
    "route",
    "authorized_by"
]].head(20)

,dispatch_id,store_id,sku,description_sap,quantity_dispatched,price_sap,dispatch_value_mxn,route,authorized_by
0,DSP-00001005,1,SAP-0002115,FUNDA APPLE 128GB,11,28059.41,308653.51,SUR,MGARCI07
1,DSP-00001183,1,SAP-0002200,"AIRPODS APPLE 75""",7,26969.63,188787.41,SUR,RCISNE04
2,DSP-00001490,2,SAP-0002234,CARGADOR APPLE 256GB,8,20342.50,162740.00,SUR,RCISNE04
3,DSP-00001895,2,SAP-0002170,"AIRPODS APPLE 65""",7,10266.10,71862.70,SUR,RCISNE04
4,DSP-00001984,2,SAP-0002103,AIRPODS APPLE 512GB,12,17422.81,209073.72,SUR,JLOPEZ11
5,DSP-00003334,3,SAP-0002127,IPHONE APPLE 512GB,4,9220.79,36883.16,GOLFO,RCISNE04
6,DSP-00003451,3,SAP-0002193,"IPHONE APPLE 32""",5,15276.40,76382.00,GOLFO,MGARCI07
7,DSP-00003523,3,SAP-0002120,"FUNDA APPLE 55""",12,26302.09,315625.08,GOLFO,MGARCI07
8,DSP-00003672,3,SAP-0002197,MOTOROLA APPLE 128GB,7,11890.28,83231.96,GOLFO,RCISNE04
9,DSP-00003735,3,SAP-0002193,"IPHONE APPLE 32""",6,15276.40,91658.40,GOLFO,RCISNE04


In [15]:
ghost_value_summary = pd.DataFrame({
    "indicador": [
        "Despachos de Ghost SKUs",
        "Tiendas alcanzadas",
        "Rutas involucradas",
        "Unidades totales despachadas",
        "Valor económico total despachado (MXN)"
    ],
    "valor": [
        len(ghost_dispatch_value),
        ghost_dispatch_value["store_id"].nunique(),
        ghost_dispatch_value["route"].nunique(),
        ghost_dispatch_value["quantity_dispatched"].sum(),
        ghost_dispatch_value["dispatch_value_mxn"].sum()
    ]
})

ghost_value_summary

,indicador,valor
0,Despachos de Ghost SKUs,427.00
1,Tiendas alcanzadas,170.00
2,Rutas involucradas,5.00
3,Unidades totales despachadas,2767.00
4,Valor económico total despachado (MXN),49936497.58


In [16]:
ghost_dispatch_value.groupby("route").agg(
    despachos=("dispatch_id", "count"),
    unidades=("quantity_dispatched", "sum"),
    valor_total_mxn=("dispatch_value_mxn", "sum")
).sort_values("valor_total_mxn", ascending=False)

,despachos,unidades,valor_total_mxn
route,,,
PACIFICO,110,764,13776326.97
GOLFO,106,669,12144313.51
CENTRO,96,587,10641774.06
SUR,80,538,9337441.11
NORTE,35,209,4036641.93


In [17]:
top_severe_store_ids = [156, 47, 15, 83, 29, 134, 67, 91, 112, 171]

ghost_to_severe = ghost_dispatch_value[
    ghost_dispatch_value["store_id"].isin(top_severe_store_ids)
].copy()

ghost_to_severe_summary = ghost_to_severe.groupby("store_id").agg(
    despachos_ghost=("dispatch_id", "count"),
    unidades_ghost=("quantity_dispatched", "sum"),
    valor_ghost_mxn=("dispatch_value_mxn", "sum")
).reset_index().sort_values("valor_ghost_mxn", ascending=False)

ghost_to_severe_summary

,store_id,despachos_ghost,unidades_ghost,valor_ghost_mxn
8,156,4,22,556059.02
0,15,5,31,458997.22
5,91,1,11,308406.34
3,67,1,12,305484.84
4,83,4,17,302000.45
6,112,2,14,294833.85
7,134,1,8,224295.52
9,171,1,8,199795.44
2,47,2,8,178316.10
1,29,2,5,58487.21


In [18]:
ghost_to_severe_summary["store_id"] = ghost_to_severe_summary["store_id"].astype(str)

fig = px.bar(
    ghost_to_severe_summary.sort_values("valor_ghost_mxn", ascending=True),
    x="valor_ghost_mxn",
    y="store_id",
    orientation="h",
    text="valor_ghost_mxn",
    title="Valor de Ghost SKUs despachados a tiendas severas"
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    marker_color="#EF4444",
    hovertemplate="<b>Tienda %{y}</b><br>Valor Ghost: $%{x:,.0f} MXN<extra></extra>"
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650,
    xaxis_title="Valor económico despachado (MXN)",
    yaxis_title="Store ID"
)

fig.show()

In [19]:
ghost_to_severe_summary["store_id"] = ghost_to_severe_summary["store_id"].astype(str)

fig = px.scatter(
    ghost_to_severe_summary,
    x="unidades_ghost",
    y="valor_ghost_mxn",
    size="despachos_ghost",
    text="store_id",
    color="valor_ghost_mxn",
    color_continuous_scale=[
        [0.00, "#3B82F6"],
        [0.35, "#06B6D4"],
        [0.70, "#8B5CF6"],
        [1.00, "#EF4444"]
    ],
    title="Exposición de Ghost SKUs en tiendas severas"
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(color="rgba(255,255,255,0.25)", width=1.2)),
    hovertemplate=(
        "<b>Tienda %{text}</b><br>" +
        "Unidades Ghost: %{x}<br>" +
        "Valor Ghost: $%{y:,.0f} MXN<br>" +
        "Despachos Ghost: %{marker.size}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14),
    height=700,
    xaxis_title="Unidades Ghost despachadas",
    yaxis_title="Valor económico Ghost (MXN)",
    coloraxis_showscale=False
)

fig.show()

## Conclusión ejecutiva

Hemos identificado un flujo material de aproximadamente **$49.9M MXN** en mercancía movilizada bajo SKUs sin trazabilidad completa en el entorno legacy, utilizando activamente la infraestructura oficial de la compañía. Esta exposición económica se superpone con las tiendas de mayor severidad operativa, lo que convierte una brecha de catálogo y control en una vulnerabilidad patrimonial y de cumplimiento de alta materialidad. La recomendación inmediata es intervenir este circuito de inventario de forma focalizada para contener la pérdida, restaurar trazabilidad y preparar una segunda fase de revisión investigativa bajo resguardo formal de evidencia.

## Conclusión ejecutiva

El análisis confirma que la opacidad de catálogo ya no es un problema técnico aislado, sino un riesgo operativo material: aproximadamente **$49.9M MXN** en mercancía se han movilizado bajo SKUs sin trazabilidad completa en AS400. Este flujo coincide con las tiendas de mayor severidad de discrepancia, lo que indica que las fallas de control y la pérdida material convergen en los mismos nodos de riesgo. La prioridad inmediata debe ser contener ese circuito, recuperar visibilidad sobre los activos premium y escalar a una segunda fase de validación forense.

## Nota estratégica — Respuesta al contraargumento de Logística

### Contraargumento esperado
Una objeción previsible desde Logística sería afirmar que la presencia de Ghost SKUs en múltiples rutas y tiendas demuestra que el hallazgo corresponde únicamente a un problema sistémico de maestro de datos, y no a una vulnerabilidad localizada.

### Respuesta analítica
Ese argumento es parcialmente correcto, pero insuficiente.

Sí existe una debilidad estructural de trazabilidad en el entorno legacy:
- los Ghost SKUs aparecen en múltiples rutas
- la opacidad del catálogo no está limitada a una sola tienda
- la falla base de control es corporativa

Sin embargo, el hallazgo principal no descansa únicamente en la existencia de esos SKUs invisibles.

La señal crítica emerge de la **convergencia de variables** que no están distribuidas de manera homogénea en toda la red:

- discrepancias materiales de recepción entre **13% y 20.5%**
- concentración completa de tiendas severas en la **Ruta Norte**
- presencia de recepciones en la franja de las **05:00**, ausente en las tiendas de comportamiento normal
- superposición entre tiendas severas y flujo de Ghost SKUs

### Conclusión metodológica
Por tanto, el análisis no sostiene que la Ruta Norte sea la única parte afectada por la debilidad del sistema.  
Lo que sostiene es algo más preciso:

> La falla de control es sistémica, pero su manifestación operativa más severa y económicamente relevante aparece localizada en un corredor específico.

### Implicación de negocio
Esto significa que la respuesta no debe ser binaria (“todo es un problema de sistema” vs “todo es un problema local”), sino secuencial:

1. reconocer la debilidad estructural corporativa
2. intervenir primero donde esa debilidad coincide con pérdida material y comportamiento atípico
3. escalar después la corrección del control a toda la red

### Riesgo de una lectura equivocada
Si se trata todo el problema como puramente sistémico, se diluye la capacidad de priorización y se corre el riesgo de distribuir recursos de forma homogénea donde el impacto no es homogéneo.

En otras palabras:

> El sistema puede estar mal en toda la empresa; la pérdida material no parece estarlo.

In [20]:
resumen_shadow_inventory = pd.DataFrame({
    "indicador": [
        "Ghost SKUs identificados",
        "Valor total del catálogo invisible (MXN)",
        "Despachos de Ghost SKUs",
        "Tiendas alcanzadas",
        "Rutas involucradas",
        "Unidades totales despachadas",
        "Valor económico total movilizado (MXN)",
        "Tiendas severas expuestas a Ghost SKUs"
    ],
    "valor": [
        len(sap_ghost_details),
        round(sap_ghost_details["price_sap"].sum(), 2),
        len(ghost_dispatch_value),
        ghost_dispatch_value["store_id"].nunique(),
        ghost_dispatch_value["route"].nunique(),
        int(ghost_dispatch_value["quantity_dispatched"].sum()),
        round(ghost_dispatch_value["dispatch_value_mxn"].sum(), 2),
        ghost_to_severe_summary["store_id"].nunique()
    ]
})

resumen_shadow_inventory

,indicador,valor
0,Ghost SKUs identificados,22.00
1,Valor total del catálogo invisible (MXN),408382.86
2,Despachos de Ghost SKUs,427.00
3,Tiendas alcanzadas,170.00
4,Rutas involucradas,5.00
5,Unidades totales despachadas,2767.00
6,Valor económico total movilizado (MXN),49936497.58
7,Tiendas severas expuestas a Ghost SKUs,10.00


## Conclusiones de la fase Shadow Inventory & Ghost SKUs

### Hallazgos principales
- Se identificó un subconjunto de SKUs visibles en SAP pero invisibles para AS400, concentrado en productos premium de alto valor.
- La invisibilidad del catálogo no es marginal: los Ghost SKUs están activos en la operación física y no solo en el maestro de datos.
- Se detectó movilización material de mercancía bajo estos códigos, con alcance multirruta y presencia en una gran cantidad de tiendas.
- Existe superposición entre el flujo de Ghost SKUs y las tiendas previamente identificadas con mayor severidad de discrepancia.
- Esto confirma que la opacidad de catálogo y la pérdida operativa no están ocurriendo en circuitos separados, sino convergiendo en nodos comunes de riesgo.

### Interpretación metodológica
El análisis no demuestra por sí mismo intencionalidad individual.  
Sí demuestra, en cambio, una vulnerabilidad estructural de trazabilidad con impacto económico material y manifestación operativa activa.

### Decisión analítica
La debilidad de catálogo debe tratarse como una capa de riesgo prioritaria dentro de la investigación, no como un defecto administrativo secundario.

## Próximo paso recomendado

La siguiente etapa del proyecto debe enfocarse en:

1. **Cruzar SKUs invisibles con usuarios autorizadores y rutas operativas**
2. **Medir la intersección entre inventario invisible, severidad de discrepancia y ventana horaria atípica**
3. **Construir una matriz de riesgo unificada por tienda, ruta y flujo económico**
4. **Preparar una narrativa ejecutiva consolidada para escalamiento a comité**

La pregunta ya no es si existe opacidad estructural, sino **qué parte de esa opacidad coincide con pérdida material y bajo qué controles ocurre**.